# Testing of Epistem default scheme workflow

In [1]:
!python -m pip install .. --quiet

import ee 
import luma_ge

# service_account_path = '../auth/ee-epstm2024.json'
# luma_ge.initialize_with_service_account(service_account_path)

ee.Authenticate()
ee.Initialize()

In [2]:
import geemap

# import region from Hadi's assets
region_name = "Sumatera"
regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

# import region from local
# aoi = geemap.shp_to_ee("../data/modular_mapping_approach/sumatra_test/Sumatra_Island_AoI.shp")

# Satellite image retrieval

In [3]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2024-01-01'
end = '2024-12-31'

landsat_data, meta = optical_reflectance.get_optical_data(
    aoi, start, end, optical_data='L9_SR', 
    compute_detailed_stats=False  # skip expensive aggregations
)

mosaic_landsat = composite.get_quality_mosaic(
    landsat_data, aoi, 
    calculate_coverage=False  # skip pixel counting
)

2026-07-22 09:30:35,827 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-07-22 09:30:35,828 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-07-22 09:30:35,829 - final_Image - INFO - final_Image creation initialized.
2026-07-22 09:30:35,830 - Reflectance_Data - INFO - Starting data fetch for Landsat 9 Operational Land Imager-2 Surface Reflectance
2026-07-22 09:30:35,831 - Reflectance_Data - INFO - Date range: 2024-01-01 to 2024-12-31
2026-07-22 09:30:35,831 - Reflectance_Data - INFO - Cloud cover threshold: 30%
2026-07-22 09:30:35,832 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-07-22 09:30:35,833 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-07-22 09:30:35,835 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-07-22 09:30:36,392 - final_Image - INFO - Creating quality mosaic from 250 images using NDVI as quality metric
2026-07-22 09:30:36

# Classification scheme
Use KLHK

In [4]:
import pandas as pd

classification_df = pd.read_csv('../data/modular_mapping_approach/LULC_INA.csv')

# Load default training data

In [5]:
from luma_ge.sample_data import SyncTrainData

trainshppath = '../data/modular_mapping_approach/sumatra_test/Sumatra_Island_Ext_Points.shp'
TrainField = 'ID'

TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=classification_df,
            aoi_geometry=aoi,
            training_shp_path=trainshppath
        )

        # Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
# TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=False)
        
        # Check sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )

TrainDataFinal = TrainDataDict.get('training_data')

2026-07-22 09:30:48,940 - luma_ge.sample_data - INFO - Loading training data from shapefile: ../data/modular_mapping_approach/sumatra_test/Sumatra_Island_Ext_Points.shp
2026-07-22 09:30:50,477 - luma_ge.sample_data - WARNING - 'kelas' field not found in training data
2026-07-22 09:30:50,478 - luma_ge.sample_data - INFO - Available columns: ['ID', 'LULC_24', 'geometry']


# Generate model classifier

In [6]:
from ee import classifier

from luma_ge.classification import FeatureExtraction

labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, mosaic_landsat, 
                            class_prop='ID', train_ratio=0.7)

# create classifier in multiprobability output mode
clf_sumatra_prob = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=1
).setOutputMode('MULTIPROBABILITY').train(
        features=strafied_train,
        classProperty='ID',
        inputProperties=mosaic_landsat.bandNames()
        )



2026-07-22 09:30:53,199 - pyogrio._io - INFO - Created 7,098 records


Stratified Random Split Training Pixel Size: 4933
Stratified Random Split Testing Pixel Size: 2165


# Classify and save MULTIPROBABILITY image stack to Google Drive

In [7]:
# rename band names of the probability
import re


def sanitize_band_name(name: str) -> str:
    """GEE band names should avoid spaces/special chars for safety downstream."""
    name = str(name).strip()
    name = re.sub(r'[^\w]+', '_', name)   # replace non-word chars with underscore
    name = re.sub(r'_+', '_', name).strip('_')
    return name

classification_df = classification_df.sort_values("ID").reset_index(drop=True)
class_labels = [sanitize_band_name(name) for name in classification_df["LULC_2024"]]

#classify probability
probability_stack = mosaic_landsat.select(mosaic_landsat.bandNames()).classify(clf_sumatra_prob)
probability_stack = probability_stack.arrayFlatten([class_labels])

bands = probability_stack.bandNames().getInfo()
samples = probability_stack.sample(region=aoi, scale=30, numPixels=5)
print(f"✓ {len(bands)} bands: {bands}")
print(f"✓ Sample values look good: {samples.getInfo()['features'][0]['properties']}")


EEException: Computation timed out.

In [8]:

# --- Export to Drive ---
task = ee.batch.Export.image.toDrive(
    image=probability_stack,
    description='sumatra_multiprobability_stack_KLHK_30m',
    folder='GEE_exports',
    fileNamePrefix='sumatra_multiprobability_stack_KLHK_30m',
    region=aoi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)
task.start()

# Probability Bands Visualization

In [ ]:
# Visualize individual probability bands
prob_map = geemap.Map()
prob_map.centerObject(aoi, 7)

# Get band names from prob_bands
band_names = bands
print(f"Available probability bands: {band_names}")

# Add each probability band to the map
prob_vis_params = {
    "min": 0,
    "max": 1,
    "palette": ['white', 'black']  # Grayscale for probability (0=white, 1=black)
}

for i, band_name in enumerate(band_names):
    band = probability_stack.select(band_name)
    prob_map.addLayer(band, prob_vis_params, f"Probability - {band_name}", shown=False)

# Also create an RGB composite using first 3 probability bands
if len(band_names) >= 3:
    rgb_composite = probability_stack.select(band_names[:3])
    prob_map.addLayer(rgb_composite, {"min": 0, "max": 1}, "RGB Composite (prob_0, prob_1, prob_2)", shown=True)

prob_map